# Question 1: Music Listeners Who Became Prime Within 30 Days

**Problem:** Find the percentage of users who listened to Music and became Prime (type = 'P') within 30 days of their join date.

**Approach:**
1. Identify users who have at least one 'Music' event
2. Left join those users with their 'P' (Prime) events
3. Calculate the difference between the Prime access date and the user's join date
4. Count users whose Prime event occurred within 30 days, then compute the percentage against all Music listeners

**Solutions below:** SQL (Cell 4) and PySpark (Cells 5–8)

In [0]:
%sql
-- ============================================================
-- Setup: Create & Populate Source Tables
-- Creates a_users (user profiles) and a_events (activity events)
-- with sample data for testing
-- ============================================================

CREATE TABLE b_sql.b_practice.a_users (
    user_id INTEGER,
    name VARCHAR(20),
    join_date DATE
);

INSERT INTO b_sql.b_practice.a_users (user_id, name, join_date)
VALUES
    (1, 'Jon',    DATE '2020-02-14'),
    (2, 'Jane',   DATE '2020-02-14'),
    (3, 'Jill',   DATE '2020-02-15'),
    (4, 'Josh',   DATE '2020-02-15'),
    (5, 'Jean',   DATE '2020-02-16'),
    (6, 'Justin', DATE '2020-02-17'),
    (7, 'Jeremy', DATE '2020-02-18');


CREATE TABLE b_sql.b_practice.a_events (
    user_id INTEGER,
    type VARCHAR(10),
    access_date DATE
);

INSERT INTO b_sql.b_practice.a_events (user_id, type, access_date)
VALUES
    (1, 'Pay',   DATE '2020-03-01'),
    (2, 'Music', DATE '2020-03-02'),
    (2, 'P',     DATE '2020-03-12'),
    (3, 'Music', DATE '2020-03-15'),
    (4, 'Music', DATE '2020-03-15'),
    (1, 'P',     DATE '2020-03-16'),
    (3, 'P',     DATE '2020-03-22');


In [0]:
%sql
-- Preview: Event Data
-- Verify inserted rows and inspect event types and dates
select * from b_sql.b_practice.a_events

In [0]:
%sql
-- Preview: User Data
-- Verify inserted rows and inspect join dates
select * from b_sql.b_practice.a_users

In [0]:
%sql
-- ============================================================
-- SQL Solution: Music Listeners Who Became Prime Within 30 Days
-- 1. music CTE        → distinct user_ids with a 'Music' event
-- 2. find_user_music  → filter a_users to only Music listeners
-- 3. muser_prime_day  → left join to 'P' events, compute datediff
-- 4. Final SELECT     → count total Music users, those with
--                       diff <= 30, and compute percentage
-- ============================================================
with music as(select distinct user_id from b_sql.b_practice.a_events where type='Music'),
find_user_music as(select * from b_sql.b_practice.a_users where user_id in (select * from music)),
muser_prime_day as (
select m.*,e.type,e.access_date,datediff(e.access_date,m.join_date) as diff from find_user_music m left  join 
b_sql.b_practice.a_events e on m.user_id=e.user_id and e.type='P')
select count(distinct user_id) as user, count(case when diff<=30 then user_id else NULL end) as within_30_prime,count(case when diff<=30 then user_id else NULL end)*100.0/count(distinct user_id) as percent from muser_prime_day


In [0]:
# ============================================================
# PySpark Setup: Import Functions & Types
# Commonly used PySpark functions for the solution below
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
# PySpark: Load & Preview Source Tables
# Read a_users and a_events from Unity Catalog into DataFrames
df_user=spark.read.table("b_sql.b_practice.a_users")
df_events=spark.read.table("b_sql.b_practice.a_events")
df_user.display()
df_events.display()


In [0]:
# PySpark: Collect Music Listener IDs
# Filter for 'Music' events, get distinct user_ids, collect to Python list
music=df_events.filter(col("type")=="Music").select("user_id").distinct().collect()
print(music)
m=[row["user_id"] for row in music]
print(m)
for i in music:
  print("user_id:", i["user_id"])



In [0]:

# ============================================================
# PySpark Solution: Music Listeners Who Became Prime Within 30 Days
# 1. Filter users to only Music listeners (using list m)
# 2. Left join to events on type == 'P' to find Prime events
# 3. Compute diff_days between access_date and join_date
# 4. Flag: user_id when diff_days < 30, else NULL
# 5. Aggregate: count distinct users, flagged users, compute %
# ============================================================

from pyspark.sql.functions import col, datediff

df_user_music = (
    df_user.alias("u")
    .filter(col("u.user_id").isin(m))
    .join(
        df_events.alias("e"),
        (col("u.user_id") == col("e.user_id")) &
        (col("e.type") == "P"),
        how="left"
    )
    .select(
        col("u.user_id"),
        col("u.name"),
        col("u.join_date"),
        col("e.type"),
        col("e.access_date"),
        datediff(
            col("e.access_date"),
            col("u.join_date")
        ).alias("diff_days")
    )
)

df_user_music.display()
df_final=df_user_music.withColumn("flag",when(col("diff_days")<30,col("user_id")).otherwise(None)).select(countDistinct("user_id").alias("count_user"),count("flag").alias("music_p_user"),round(count("flag")*100.0/(countDistinct("user_id")),2).alias("percent_p")).display()